In [5]:
import librosa
from os import path

SOURCE_PATH = path.normpath("../datasets/pcm_s16le_441")
TARGET_PATH = path.normpath("../features/pcm_s16le_441")

AUDIO_SAMPLE_RATE = 44_100
CQT_HOP_LENGTH = 512

# 3 octaves in total
CQT_FEATURE_BINS = 12 * 3

# Sample rate 48 khz * 2 seconds / 512 default hop size = 187.5
CQT_FEATURE_FRAMES = 200

# First note of the feature.
# Pad -1 and +1 octave from target octave 4
CQT_FMIN = librosa.note_to_hz('C3')

In [6]:
from os import listdir, makedirs, path
from tqdm.notebook import tqdm
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

def target_npz_path(dataset_type):
    filename = f"{path.basename(TARGET_PATH)}-{dataset_type}.npz"
    return path.join(path.dirname(TARGET_PATH), filename)

def preprocess(cqt):
    """Truncates frames and zero pads features to frame length"""
    if cqt.shape[1] < CQT_FEATURE_FRAMES:
        pad_width = CQT_FEATURE_FRAMES - cqt.shape[1]
        cqt = np.pad(cqt, pad_width=((0, 0), (0, pad_width)), mode="constant")
    else:
        cqt = cqt[:, :CQT_FEATURE_FRAMES]
    return cqt

def extract_cqt_features(audio_path):
    """Extracts CQT features from an audio file."""
    try:
        y, sr = librosa.load(audio_path, sr=AUDIO_SAMPLE_RATE)
        cqt = librosa.cqt(
            y=y,
            sr=sr,
            fmin=CQT_FMIN,
            n_bins=CQT_FEATURE_BINS,
            bins_per_octave=12,
            hop_length=CQT_HOP_LENGTH,
        )
        cqt = np.abs(cqt)
        return preprocess(cqt)
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

makedirs(path.dirname(TARGET_PATH), exist_ok=True)

for dataset_type in tqdm(listdir(SOURCE_PATH), desc="Processing datasets"):
    type_path = path.join(SOURCE_PATH, dataset_type)
    if not path.isdir(type_path):
        continue

    features = []
    labels = []

    for class_name in tqdm(listdir(type_path), desc=f"Processing {dataset_type} classes"):
        class_path = path.join(type_path, class_name)
        if not path.isdir(class_path):
            continue

        for audio_file in tqdm(listdir(class_path), desc=f"Processing {class_name}"):
            audio_path = path.join(class_path, audio_file)
            if not path.isfile(audio_path):
                continue

            cqt = extract_cqt_features(audio_path)
            if cqt is not None:
                features.append(cqt)
                labels.append(class_name)

    npz_path = target_npz_path(dataset_type)
    features = np.array(features)
    labels = np.array(labels)
    np.savez(npz_path, features=features, labels=labels)
    print(f"Shape of CQT features: {features.shape}")
    print(f"Shape of CQT labels: {labels.shape}")
    print(f"Features and labels saved to {npz_path}")

Processing datasets:   0%|          | 0/2 [00:00<?, ?it/s]

Processing clean classes:   0%|          | 0/36 [00:00<?, ?it/s]

Processing A#_diminished_4:   0%|          | 0/200 [00:00<?, ?it/s]

Processing A#_major_4:   0%|          | 0/200 [00:00<?, ?it/s]

Processing A#_minor_4:   0%|          | 0/200 [00:00<?, ?it/s]

...

Processing G#_minor_4:   0%|          | 0/100 [00:00<?, ?it/s]

Processing G_diminished_4:   0%|          | 0/100 [00:00<?, ?it/s]

Processing G_major_4:   0%|          | 0/100 [00:00<?, ?it/s]

Processing G_minor_4:   0%|          | 0/100 [00:00<?, ?it/s]

Shape of CQT features: (3600, 36, 200)
Shape of CQT labels: (3600,)
Features and labels saved to ../features/pcm_s16le_441-noisy.npz
